# Lab 03: the overhead floor

Batch-1 step time with kernels launched one by one and with the launch sequence replayed, on the 4B model and the 0.6B model, beside the bytes-over-bandwidth prediction. Each configuration loads its own engine; run the cells one at a time and restart the kernel between them if memory is not freed.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Labs 08 and 10 are read-only on this GPU (no FP8 arithmetic).")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: a measuring function. Restart the kernel (Run → Restart) between configurations if the GPU memory is not released.
import time, gc, torch
from vllm import LLM, SamplingParams
SPEC_GBPS = 320
def measure(model, eager, weight_gb):
    llm = LLM(model, dtype="half", max_model_len=4096, gpu_memory_utilization=0.9, enforce_eager=eager, enable_prefix_caching=False, seed=0)
    p = ["Write a short paragraph about the number 7."]
    llm.generate(p, SamplingParams(max_tokens=8, temperature=0, ignore_eos=True))
    t1 = time.time(); llm.generate(p, SamplingParams(max_tokens=1, temperature=0, ignore_eos=True)); t_one = time.time() - t1
    t1 = time.time(); llm.generate(p, SamplingParams(max_tokens=257, temperature=0, ignore_eos=True)); t_many = time.time() - t1
    step = (t_many - t_one) / 256 * 1000; pred = weight_gb / SPEC_GBPS * 1000
    print(f"{model:<26} {'one by one' if eager else 'replayed':<12} predicted {pred:5.1f} ms  measured {step:5.1f} ms  floor {step - pred:5.1f} ms")
    del llm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
measure("Qwen/Qwen3-4B-Instruct-2507", True, 8.04)

In [ ]:
measure("Qwen/Qwen3-4B-Instruct-2507", False, 8.04)

In [ ]:
measure("Qwen/Qwen3-0.6B", True, 1.50)

In [ ]:
measure("Qwen/Qwen3-0.6B", False, 1.50)